# Build panel DataFrame (issue #72)

Restructure `master_df` into a `(date, ticker)` MultiIndex panel for panel-style analysis.
Mirrors `scripts/build_panel_df.py` and `src/macro_regime/panel.py`.

In [ ]:
import pandas as pd
import os
print(os.getcwd())

In [ ]:
processed_dir = "../data/processed"

master = pd.read_csv(f"{processed_dir}/master_df.csv", parse_dates=["date"])
master.head()

In [ ]:
# Drop redundant name column (ticker -> name is 1-to-1), set MultiIndex, sort
panel = master.drop(columns=["name"]).set_index(["date", "ticker"]).sort_index()
panel.head()

## Panel slicing

- Cross-section: all tickers on one date
- Time series: one ticker across all dates
- Single cell: one (date, ticker) pair

In [ ]:
# Cross-section: all tickers on one date
panel.loc["2024-01-02"]

In [ ]:
# Time series: one ticker across all dates
panel.xs("XLE", level="ticker").head()

In [ ]:
# Single cell: one (date, ticker) pair
panel.loc[("2024-01-02", "XLE")]

In [ ]:
# Wide pivot: one column per sector ticker (typical panel-analysis view)
wide = panel["sector_return"].unstack("ticker")
wide.tail()

In [ ]:
print(f"Shape: {panel.shape}")
print(f"Index levels: {list(panel.index.names)}")
print(f"Monotonic increasing: {panel.index.is_monotonic_increasing}")
print(f"Unique index: {panel.index.is_unique}")
print("\nMissing values:")
print(panel.isnull().sum())

In [ ]:
panel.to_csv(f"{processed_dir}/master_panel_df.csv", index=True)
print("Saved to data/processed/master_panel_df.csv")
print("Reload: pd.read_csv(path, index_col=[0, 1], parse_dates=['date'])")